# Scalability with Prefect and Ray: Multi-Region Bayesian Inference

This notebook validates ProbPipe's Prefect integration by distributing a realistic workload across workers. The scenario is **multi-region Poisson regression**: the same model is fitted independently to data from many regions, making it perfect for parallel computation.

We use two complementary approaches:

1. **Broadcasting via `WorkflowFunction`** with `workflow_kind="task"`: ProbPipe's built-in Prefect integration distributes per-region MCMC runs as Prefect tasks.
2. **Posterior predictive broadcasting**: once posteriors are collected, we propagate uncertainty through a prediction function, again distributed via Prefect.

For true parallel execution, we use **Ray** as the Prefect task runner.

**Prerequisites:**
```bash
pip install probpipe[prefect] prefect-ray ray
```

This notebook builds on concepts from:
- [04_broadcasting.ipynb](04_broadcasting.ipynb) — WorkflowFunction and uncertainty propagation
- [06_modular_forecasting.ipynb](06_modular_forecasting.ipynb) — Bayesian inference with `condition_on`

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=r"Explicitly requested dtype.*float64.*")

import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from probpipe import (
    Normal, MultivariateNormal, EmpiricalDistribution,
    SimpleModel, condition_on, from_distribution,
    sample as pp_sample, mean,
)
from probpipe.core.node import WorkflowFunction, workflow_function

rng = np.random.default_rng(42)

## Problem Setup: Multi-Region Poisson Regression

Imagine modelling disease counts across **20 independent regions**. Each region has its own covariate data and observed counts, but we fit the same model structure to each:

$$y_{ij} \sim \text{Poisson}\bigl(\exp(\beta_{0,i} + \beta_{1,i} \, x_{ij})\bigr)$$

where $i$ indexes region and $j$ indexes observations within a region. Since the regions are independent, their MCMC fits can run in parallel.

In [ ]:
N_REGIONS = 20
N_OBS_PER_REGION = 150

# True parameters vary by region (drawn from a shared hyper-distribution)
true_betas = np.column_stack([
    rng.normal(1.0, 0.3, size=N_REGIONS),   # intercepts ~ Normal(1, 0.3)
    rng.normal(0.5, 0.2, size=N_REGIONS),   # slopes ~ Normal(0.5, 0.2)
]).astype(np.float32)

# Generate covariate and count data for each region
region_data = []
for i in range(N_REGIONS):
    x = rng.uniform(-2, 2, size=N_OBS_PER_REGION).astype(np.float32)
    rates = np.exp(true_betas[i, 0] + true_betas[i, 1] * x)
    y = rng.poisson(rates).astype(np.float32)
    region_data.append((x, y))

print(f"Regions: {N_REGIONS}")
print(f"Observations per region: {N_OBS_PER_REGION}")
print(f"True beta ranges: intercept [{true_betas[:, 0].min():.2f}, {true_betas[:, 0].max():.2f}], "
      f"slope [{true_betas[:, 1].min():.2f}, {true_betas[:, 1].max():.2f}]")

## Likelihood and Inference Function

We define a function that takes a region index, builds the model, runs MCMC via `condition_on`, and returns the posterior mean. This function will be the unit of work that Prefect distributes.

In [ ]:
class PoissonRegressionLikelihood:
    """Poisson regression: y_i ~ Poisson(exp(beta_0 + beta_1 * x_i))."""

    def __init__(self, x: np.ndarray):
        self._x = jnp.asarray(x, dtype=jnp.float32)

    def log_likelihood(self, params, data):
        log_rate = params[0] + params[1] * self._x
        rate = jnp.exp(log_rate)
        return jnp.sum(data * log_rate - rate)


# Shared prior for all regions
PRIOR = MultivariateNormal(
    loc=jnp.zeros(2), cov=10.0 * jnp.eye(2), name="shared_prior"
)


def fit_region(region_idx: jnp.ndarray) -> jnp.ndarray:
    """Run MCMC for a single region and return the posterior mean.

    Parameters
    ----------
    region_idx : scalar array
        Index into ``region_data`` (converted to int internally).

    Returns
    -------
    jnp.ndarray, shape (2,)
        Posterior mean of (beta_0, beta_1) for this region.
    """
    idx = int(region_idx)
    x, y = region_data[idx]
    likelihood = PoissonRegressionLikelihood(x)
    model = SimpleModel(PRIOR, likelihood)
    posterior = condition_on(
        model, y, num_results=500, num_warmup=300, random_seed=idx,
    )
    return jnp.array(mean(posterior))

## Baseline: Sequential Execution

First, run all 20 region fits sequentially to establish a timing baseline.

In [ ]:
t0 = time.perf_counter()
sequential_means = []
for i in range(N_REGIONS):
    posterior_mean = fit_region(jnp.array(i))
    sequential_means.append(posterior_mean)
t_sequential = time.perf_counter() - t0

sequential_means = jnp.stack(sequential_means)
print(f"Sequential: {t_sequential:.1f}s for {N_REGIONS} regions")
print(f"Average per region: {t_sequential / N_REGIONS:.2f}s")

## Prefect-Distributed Execution via WorkflowFunction

Now we use ProbPipe's built-in Prefect integration. The strategy:

1. Create an `EmpiricalDistribution` whose "samples" are region indices `[0, 1, ..., 19]`.
2. Wrap `fit_region` in a `WorkflowFunction` with `workflow_kind="task"`.
3. When the function receives the empirical distribution, it **enumerates** all 20 indices and dispatches each as a Prefect task via `task.map()`.

This leverages the existing broadcasting + Prefect plumbing in `WorkflowFunction._execute_many`.

In [ ]:
# Region indices as an EmpiricalDistribution — each "sample" is a region ID
region_indices = EmpiricalDistribution(
    jnp.arange(N_REGIONS, dtype=jnp.float32).reshape(-1, 1),
    name="region_indices",
)

# WorkflowFunction with Prefect task dispatch
# vectorize="loop" because fit_region is not JAX-traceable (it runs MCMC internally)
fit_region_wf = WorkflowFunction(
    func=fit_region,
    workflow_kind="task",
    vectorize="loop",
    n_broadcast_samples=N_REGIONS,
    seed=0,
    name="fit_region",
)

print(f"Region indices: {region_indices.n} samples, event_shape={region_indices.event_shape}")
print(f"Workflow kind: {fit_region_wf._workflow_kind}")
print(f"Vectorize: {fit_region_wf._vectorize}")

In [ ]:
t0 = time.perf_counter()
result = fit_region_wf(region_idx=region_indices)
t_prefect = time.perf_counter() - t0

print(f"Prefect task.map(): {t_prefect:.1f}s for {N_REGIONS} regions")
print(f"Sequential was: {t_sequential:.1f}s")
print(f"Ratio: {t_prefect / t_sequential:.2f}x")

## Validate Correctness

The Prefect-distributed results should match the sequential baseline exactly (same seeds, same data).

In [ ]:
prefect_means = result.samples  # shape (N_REGIONS, 2)

max_diff = float(jnp.max(jnp.abs(prefect_means - sequential_means)))
print(f"Max absolute difference: {max_diff:.6f}")
print(f"Results match: {max_diff < 1e-4}")

# Compare a few regions
print(f"\n{'Region':<8} {'Sequential':<24} {'Prefect':<24} {'True'}")
print("-" * 80)
for i in [0, 5, 10, 15, 19]:
    s = sequential_means[i]
    p = prefect_means[i]
    t = true_betas[i]
    print(f"{i:<8} [{s[0]:.3f}, {s[1]:.3f}]{'':>8} [{p[0]:.3f}, {p[1]:.3f}]{'':>8} [{t[0]:.3f}, {t[1]:.3f}]")

## Provenance Tracking

The distributed result preserves full provenance — which function, which distribution arguments, and the orchestration strategy used.

In [ ]:
from probpipe import provenance_ancestors

print(f"Source: {result.source}")
print(f"Operation:    {result.source.operation}")
print(f"Orchestrate:  {result.source.metadata['orchestrate']}")
print(f"Vectorize:    {result.source.metadata['vectorize']}")
print(f"Function:     {result.source.metadata['func']}")
print(f"N samples:    {result.source.metadata['n_samples']}")

ancestors = provenance_ancestors(result)
print(f"\nAncestors: {[a.name for a in ancestors]}")

## Visualize Results Across Regions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

regions = np.arange(N_REGIONS)

# Intercepts
ax1.scatter(regions, true_betas[:, 0], marker='x', color='red', s=60, label='True', zorder=3)
ax1.scatter(regions, np.array(prefect_means[:, 0]), marker='o', color='steelblue',
            s=40, alpha=0.7, label='Posterior mean')
ax1.set_xlabel('Region')
ax1.set_ylabel(r'$\beta_0$ (intercept)')
ax1.set_title('Intercept estimates by region')
ax1.legend()

# Slopes
ax2.scatter(regions, true_betas[:, 1], marker='x', color='red', s=60, label='True', zorder=3)
ax2.scatter(regions, np.array(prefect_means[:, 1]), marker='o', color='orange',
            s=40, alpha=0.7, label='Posterior mean')
ax2.set_xlabel('Region')
ax2.set_ylabel(r'$\beta_1$ (slope)')
ax2.set_title('Slope estimates by region')
ax2.legend()

plt.suptitle(f'Multi-region Poisson regression ({N_REGIONS} regions, Prefect-distributed)', y=1.02)
plt.tight_layout()
plt.show()

## Posterior Predictive Broadcasting with Prefect

Next, we test the second Prefect path: broadcasting posterior samples through a prediction function. This is useful for generating predictions with full uncertainty quantification.

We take the posterior from one region, sample parameters from it, and compute predicted rates across a grid of covariate values — each evaluation dispatched as a Prefect task.

In [ ]:
# Get full posterior for region 0
x0, y0 = region_data[0]
likelihood0 = PoissonRegressionLikelihood(x0)
model0 = SimpleModel(PRIOR, likelihood0)
posterior0 = condition_on(model0, y0, num_results=1000, num_warmup=500, random_seed=0)

# Fit a Gaussian approximation for broadcasting
posterior0_approx = from_distribution(posterior0, MultivariateNormal, name="posterior_region0")
print(f"Posterior approx: {posterior0_approx}")
print(f"Mean: {mean(posterior0_approx)}")

In [ ]:
x_grid = jnp.linspace(-2, 2, 50)


def predict_rates(params: jnp.ndarray) -> jnp.ndarray:
    """Compute predicted rates across the covariate grid."""
    return jnp.exp(params[0] + params[1] * x_grid)


# Sequential baseline
predict_seq = WorkflowFunction(
    func=predict_rates,
    n_broadcast_samples=500,
    vectorize="loop",
    seed=0,
    name="predict_rates",
)

t0 = time.perf_counter()
rates_seq = predict_seq(params=posterior0_approx)
t_seq = time.perf_counter() - t0

# Prefect task-distributed
predict_prefect = WorkflowFunction(
    func=predict_rates,
    workflow_kind="task",
    n_broadcast_samples=500,
    vectorize="loop",
    seed=0,
    name="predict_rates",
)

t0 = time.perf_counter()
rates_prefect = predict_prefect(params=posterior0_approx)
t_prefect_pred = time.perf_counter() - t0

print(f"Posterior predictive (500 samples x 50 grid points):")
print(f"  Sequential: {t_seq:.3f}s")
print(f"  Prefect:    {t_prefect_pred:.3f}s")
print(f"  Ratio:      {t_prefect_pred / t_seq:.2f}x")

In [ ]:
# Verify results match
max_diff = float(jnp.max(jnp.abs(rates_seq.samples - rates_prefect.samples)))
print(f"Max difference between sequential and Prefect: {max_diff:.6f}")

# Plot posterior predictive
rate_samples = np.array(rates_prefect.samples)
rate_mean = rate_samples.mean(axis=0)
rate_lower = np.percentile(rate_samples, 5, axis=0)
rate_upper = np.percentile(rate_samples, 95, axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x0, y0, alpha=0.3, s=15, color='steelblue', label='Observed counts')
ax.plot(x_grid, np.exp(true_betas[0, 0] + true_betas[0, 1] * np.array(x_grid)),
        'r--', linewidth=2, label='True rate')
ax.plot(x_grid, rate_mean, 'k-', linewidth=2, label='Posterior mean rate')
ax.fill_between(np.array(x_grid), rate_lower, rate_upper, alpha=0.2, color='gray',
                label='90% credible interval')
ax.set_xlabel('x')
ax.set_ylabel('count / rate')
ax.set_title('Posterior predictive — Region 0 (Prefect-distributed)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Scaling: Varying Region Count

To see how the overhead scales, we run the multi-region fit with increasing numbers of regions and compare sequential vs. Prefect timing.

In [ ]:
region_counts = [5, 10, 20]
timing_results = []

for n_reg in region_counts:
    # Sequential
    t0 = time.perf_counter()
    for i in range(n_reg):
        fit_region(jnp.array(i))
    t_seq = time.perf_counter() - t0

    # Prefect
    indices = EmpiricalDistribution(
        jnp.arange(n_reg, dtype=jnp.float32).reshape(-1, 1),
    )
    wf = WorkflowFunction(
        func=fit_region,
        workflow_kind="task",
        vectorize="loop",
        n_broadcast_samples=n_reg,
        seed=0,
        name="fit_region",
    )
    t0 = time.perf_counter()
    wf(region_idx=indices)
    t_pf = time.perf_counter() - t0

    timing_results.append((n_reg, t_seq, t_pf))
    print(f"n_regions={n_reg:>3}: sequential={t_seq:.1f}s, prefect={t_pf:.1f}s, ratio={t_pf/t_seq:.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ns = [r[0] for r in timing_results]
t_seqs = [r[1] for r in timing_results]
t_pfs = [r[2] for r in timing_results]

ax.plot(ns, t_seqs, 'o-', color='steelblue', linewidth=2, markersize=8, label='Sequential')
ax.plot(ns, t_pfs, 's-', color='orange', linewidth=2, markersize=8, label='Prefect task.map()')
ax.set_xlabel('Number of regions')
ax.set_ylabel('Wall time (s)')
ax.set_title('Scaling: Sequential vs. Prefect')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Distributed Execution with Ray

The examples above use Prefect's **default task runner**, which executes tasks sequentially in-process. To actually distribute across workers or machines, we use **Ray** as the task runner.

### Setup

```bash
pip install prefect-ray ray
```

### Using RayTaskRunner

Ray can run tasks in parallel across local CPU cores or across a cluster of machines:

```python
from prefect_ray import RayTaskRunner

# Local: uses all available CPU cores
runner = RayTaskRunner()

# Remote: connect to an existing Ray cluster
runner = RayTaskRunner(address="ray://cluster-head:10001")
```

### Integrating with WorkflowFunction

Currently, `WorkflowFunction` creates its own Prefect flow internally (see `_execute_many_prefect_task`). To use Ray as the task runner, the flow must be configured with the runner. This requires a small extension to `WorkflowFunction` — passing a `task_runner` parameter through to the internally-created flow:

```python
from prefect_ray import RayTaskRunner

# Proposed API extension:
fit_region_wf = WorkflowFunction(
    func=fit_region,
    workflow_kind="task",
    task_runner=RayTaskRunner(),
    vectorize="loop",
    n_broadcast_samples=N_REGIONS,
    seed=0,
)
```

With this extension, each region's MCMC fit would run as a Ray remote task — truly parallel across CPU cores (or machines in a cluster). For 20 regions on an 8-core machine, this could yield up to ~8x speedup over sequential execution.

### Ray Cluster Deployment

For multi-machine scaling:

```bash
# On the head node:
ray start --head --port=6379

# On worker nodes:
ray start --address="head-node-ip:6379"
```

Then connect the task runner:
```python
runner = RayTaskRunner(address="ray://head-node-ip:10001")
```

This enables distributing MCMC fits across a cluster with no changes to the inference code — only the task runner configuration changes.